# Phase 6: Model Evaluation & Explainability

In this phase we analyze and interpret the churn prediction model.

Objectives:
- Evaluate model performance using advanced metrics
- Analyze prediction quality using ROC curves
- Understand feature importance
- Interpret drivers of customer churn

In [ ]:
# ============================================
# 1. Imports
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay,
    precision_recall_curve,
    PrecisionRecallDisplay
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

In [ ]:
# ============================================
# 2. Load Customer Dataset
# ============================================

customer_df = pd.read_csv("../data/processed/customer_dataset.csv")

print("Dataset Shape:", customer_df.shape)
customer_df.head()

## Recreate Modeling Dataset

To evaluate the model independently, we recreate
the churn label and modeling dataset.

In [ ]:
# ============================================
# 3. Create Churn Label
# ============================================

churn_threshold = 90

customer_df['churn'] = np.where(
    customer_df['recency_days'] > churn_threshold,
    1,
    0
)

In [ ]:
customer_df['avg_order_value'] = customer_df['avg_order_value'].fillna(
    customer_df['avg_order_value'].median()
)

customer_df['max_order_value'] = customer_df['max_order_value'].fillna(
    customer_df['max_order_value'].median()
)

customer_df['avg_review_score'] = customer_df['avg_review_score'].fillna(
    customer_df['avg_review_score'].median()
)

In [ ]:
# ============================================
# Feature Selection
# ============================================

features = customer_df[
    [
        'recency_days',
        'frequency',
        'monetary',
        'avg_order_value',
        'avg_review_score'
    ]
]

target = customer_df['churn']

In [ ]:
# ============================================
# Train Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42
)

In [ ]:
# ============================================
# Train Random Forest Model
# ============================================

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:,1]

## Model Performance Evaluation

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:\n", cm)

## ROC Curve Analysis

The ROC curve evaluates the model's ability
to distinguish churned and active customers.

In [ ]:
RocCurveDisplay.from_estimator(
    rf_model,
    X_test,
    y_test
)

plt.title("ROC Curve")
plt.show()

In [ ]:
roc_score = roc_auc_score(y_test, y_prob)

print("ROC-AUC Score:", roc_score)

## Precision Recall Analysis

Precision-Recall curves are useful when
class imbalance exists.

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_prob)

PrecisionRecallDisplay(
    precision=precision,
    recall=recall
).plot()

plt.title("Precision-Recall Curve")
plt.show()

## Feature Importance Analysis

Feature importance helps identify
which variables influence churn prediction most.

In [ ]:
importance = pd.Series(
    rf_model.feature_importances_,
    index=features.columns
)

importance.sort_values().plot(kind='barh')

plt.title("Feature Importance for Churn Prediction")

plt.show()

In [ ]:
importance.sort_values(ascending=False)

# Model Explainability Insights

Key Observations:

• Recency is the strongest predictor of churn.
• Customers with lower purchase frequency tend to churn.
• Lower monetary contribution is associated with higher churn risk.

Business Implications:

• Monitor customers with increasing inactivity.
• Target low-frequency buyers with re-engagement campaigns.
• Reward loyal customers to maintain retention.